# Fase 1C — IOI Circuit

Reprodução do protocolo de Wang et al. (2022) para identificação do circuito de
Indirect Object Identification (IOI) em GPT-2 small via activation patching.

**Referência:** Wang, K. et al. (2022). *Interpretability in the Wild: a Circuit for
Indirect Object Identification in GPT-2 small.* arXiv:2211.00593.

**Tarefa:** dado o prompt *"When Mary and John went to the store, John gave a bottle of milk to"*,
o modelo deve prever *Mary* (objeto indireto) e não *John* (sujeito).

**Critério de validação:** as Name Mover Heads com maior score normalizado devem incluir
heads nas camadas 9-10, consistentes com Wang et al. (L9H6, L9H9, L10H0).

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

from src.inspector.model_inspector import ModelInspector, InspectorConfig
from src.circuits.ioi_utils import (
    load_ioi_prompts,
    ioi_accuracy,
    compute_patching_map,
    top_name_mover_heads,
    corrupt_prompt,
    logit_diff,
)

torch.manual_seed(42)

## 1. Carregar modelo e dataset

In [ ]:
inspector = ModelInspector(InspectorConfig(model_name='gpt2', device='auto'))
inspector.load()

prompts = load_ioi_prompts()
print(f'Modelo: GPT-2 small | Prompts IOI: {len(prompts)}')
print(f'\nExemplo de prompt:')
print(f'  Clean:     {prompts[0]["prompt"]}')
print(f'  Corrupted: {corrupt_prompt(prompts[0])}')
print(f'  Correto:   "{prompts[0]["correct_token"]}" | Incorreto: "{prompts[0]["incorrect_token"]}"')

## 2. Acurácia baseline

GPT-2 small deve resolver a tarefa IOI com ~85% de acurácia.

In [ ]:
result = ioi_accuracy(inspector, prompts)

print(f'Acurácia IOI: {result["accuracy"]:.1%}')
print(f'Logit diff média: {result["mean_logit_diff"]:.3f}')
print(f'\nLogit diffs por prompt:')
for p, d in zip(prompts, result['logit_diffs']):
    status = 'CORRECT' if d > 0 else 'WRONG'
    io = p['indirect_object']
    s = p['subject']
    print(f'  [{status}] {io} vs {s}: {d:+.3f}')

## 3. Activation Patching — mapa completo

Para cada head e MLP, substituímos a ativação pela versão do prompt corrompido
(nomes trocados) e medimos o impacto na logit diff.

**Score normalizado:**
- 1.0 → componente recupera toda a performance (componente crucial)
- 0.0 → componente não afeta a resposta
- < 0 → componente prejudica a resposta correta

> **Atenção:** este cálculo faz (n_layers × n_heads + n_layers) × n_prompts forward passes.
> Com 25 prompts e GPT-2 small: ~3900 forward passes. Pode levar 5-15 min na GPU.

In [ ]:
# Usar subconjunto para agilizar; aumentar para mais precisão
PROMPTS_SUBSET = prompts[:15]

print(f'Calculando patching map com {len(PROMPTS_SUBSET)} prompts...')
patching_map = compute_patching_map(inspector, PROMPTS_SUBSET)
print(f'Pronto.')
print(f'  Baseline (clean):     {patching_map["baseline"]:.3f}')
print(f'  Baseline (corrupted): {patching_map["baseline_corrupted"]:.3f}')

## 4. Heatmap: contribuição de cada attention head

In [ ]:
attn = patching_map['attn'].cpu().numpy()

fig = px.imshow(
    attn,
    title='Activation Patching — Contribuição das Attention Heads (normalizada)',
    labels={'x': 'Head', 'y': 'Layer', 'color': 'Score norm.'},
    x=[f'H{h}' for h in range(inspector.n_heads)],
    y=[f'L{l}' for l in range(inspector.n_layers)],
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    aspect='auto',
)

# Destaque das Name Mover Heads do paper (L9H6, L9H9, L10H0)
paper_heads = [(9, 6), (9, 9), (10, 0)]
for (l, h) in paper_heads:
    fig.add_annotation(x=h, y=l, text='★', showarrow=False,
                       font=dict(size=14, color='black'))

fig.show()

## 5. Contribuição dos MLPs

In [ ]:
mlp = patching_map['mlp'].cpu().numpy()

fig = go.Figure(go.Bar(
    x=[f'L{l}' for l in range(inspector.n_layers)],
    y=mlp,
    marker_color=['#d62728' if v > 0.1 else '#aec7e8' for v in mlp],
))
fig.update_layout(
    title='Contribuição dos MLPs por camada (score normalizado)',
    xaxis_title='Layer',
    yaxis_title='Score normalizado',
)
fig.show()

## 6. Top Name Mover Heads

In [ ]:
top_heads = top_name_mover_heads(patching_map, n_top=10)

df = pd.DataFrame(top_heads, columns=['Layer', 'Head', 'Score'])
df['Head ID'] = df.apply(lambda r: f'L{int(r.Layer)}H{int(r.Head)}', axis=1)
df['Score'] = df['Score'].round(4)

print('=== Top 10 Name Mover Heads (por score normalizado) ===')
print(df[['Head ID', 'Score']].to_string(index=False))

## 7. Comparação com Wang et al. (2022)

In [ ]:
# Name Mover Heads reportadas em Wang et al. para GPT-2 small
PAPER_HEADS = {(9, 6), (9, 9), (10, 0)}

attn_tensor = patching_map['attn']
detected_top5 = {(int(l), int(h)) for l, h, _ in top_heads[:5]}
overlap = PAPER_HEADS & detected_top5

print(f'Name Mover Heads do paper:    {sorted(PAPER_HEADS)}')
print(f'Top-5 detectadas (patching):  {sorted(detected_top5)}')
print(f'Sobreposição: {sorted(overlap)}')
print()
print('Scores das heads do paper:')
for (l, h) in sorted(PAPER_HEADS):
    score = attn_tensor[l, h].item()
    print(f'  L{l}H{h}: {score:.4f}')

## 8. Conclusão — PASS / FAIL

In [ ]:
# Critérios:
# 1. Acurácia baseline >= 70%
# 2. Pelo menos 1 das 3 Name Mover Heads do paper está no top-5 detectado
# 3. Score normalizado de pelo menos 1 head do paper >= 0.3

MIN_ACCURACY = 0.70
MIN_OVERLAP = 1
MIN_SCORE = 0.3

paper_scores = [attn_tensor[l, h].item() for l, h in PAPER_HEADS]

c1 = result['accuracy'] >= MIN_ACCURACY
c2 = len(overlap) >= MIN_OVERLAP
c3 = any(s >= MIN_SCORE for s in paper_scores)

if c1 and c2 and c3:
    print('RESULT: PASS')
else:
    print('RESULT: FAIL')

print(f'\n  [{'OK' if c1 else 'FAIL'}] Acurácia: {result["accuracy"]:.1%} (mínimo {MIN_ACCURACY:.0%})')
print(f'  [{'OK' if c2 else 'FAIL'}] Sobreposição: {len(overlap)} heads (mínimo {MIN_OVERLAP})')
print(f'  [{'OK' if c3 else 'FAIL'}] Score máx. paper heads: {max(paper_scores):.4f} (mínimo {MIN_SCORE})')